In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/squad-dataset/train.json


In [6]:
import json
import pandas as pd
import numpy as np
import math

from sklearn.feature_extraction.text import TfidfVectorizer

!pip install transformers datasets
!pip install faiss-cpu



import faiss

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
bigframes 1.42.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.9.0.13 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cudnn-cu12==9.1.0.70; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cudnn-cu12 9.3.0.75 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cufft-

In [10]:
# Load SQuAD data
file_path = 'train.json'

def convert_squad_data_to_dataframe(file_path, cnt):
    with open(file_path, 'r') as f:
        squad_dict = json.load(f)
        data = squad_dict['data']
        rows = []
        for article in data:
            for paragraph in article['paragraphs']:
                context = paragraph['context']
                for qa in paragraph['qas']:
                    question = qa['question']
                    qid = qa['id']
                    is_impossible = qa.get('is_impossible', False)
                    answers = qa['answers'] if not is_impossible else []
                    if answers:
                        for answer in answers:
                            rows.append({
                                'context': context,
                                'question': question,
                                'qid': qid,
                                'answer': answer['text'],
                                'answer_start': answer['answer_start'],
                                'c_id': cnt
                            })
                    else:
                        rows.append({
                            'context': context,
                            'question': question,
                            'qid': qid,
                            'answer': '',
                            'answer_start': -1,
                            'c_id': cnt
                        })
                cnt += 1
        return pd.DataFrame(rows)



In [11]:
cnt = 0
data = convert_squad_data_to_dataframe(file_path, cnt)

# Drop duplicate contexts
documents = data[['context', 'c_id']].drop_duplicates().reset_index(drop=True)

# Create mapping for questions to their correct context IDs
questionContext = dict(zip(data['question'], data['c_id']))

# Vectorize contexts using TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(documents['context'])

# Convert sparse to dense and normalize for FAISS
X_dense = X.toarray().astype(np.float32)
faiss.normalize_L2(X_dense)

# Build FAISS index
index = faiss.IndexFlatIP(X_dense.shape[1])
index.add(X_dense)

# Compute metrics
def dcg(scores):
    return sum([rel / math.log2(idx + 2) for idx, rel in enumerate(scores)])

def compute_metrics(question, questionContext, index, k=10, max_samples=2000):
    total_correct = 0
    reciprocal_ranks = []
    ndcg_scores = []
    precision_scores = []
    recall_scores = []
    mar_scores = []

    for idx, ques in enumerate(question):
        if idx >= max_samples:
            break
        query_vector = vectorizer.transform([ques]).toarray().astype(np.float32)
        faiss.normalize_L2(query_vector)
        distances, indices = index.search(query_vector, k)

        predictions = indices[0]
        actual_index = questionContext[ques]

        relevance = [1 if pred == actual_index else 0 for pred in predictions]

        if actual_index in predictions:
            total_correct += 1
            rank = list(predictions).index(actual_index)
            reciprocal_ranks.append(1.0 / (rank + 1))
            recall_scores.append(1.0)
            precision_scores.append(1.0 / (rank + 1))
        else:
            reciprocal_ranks.append(0.0)
            recall_scores.append(0.0)
            precision_scores.append(0.0)

        ideal_relevance = sorted(relevance, reverse=True)
        ndcg = dcg(relevance) / dcg(ideal_relevance) if sum(ideal_relevance) > 0 else 0.0
        ndcg_scores.append(ndcg)

        num_relevant_items = sum(relevance)
        mar_scores.append(num_relevant_items / len(relevance) if num_relevant_items > 0 else 0.0)

    num_samples = min(len(question), max_samples)

    return {
        f"Accuracy@{k}": (total_correct / num_samples) * 100,
        f"MRR@{k}": np.mean(reciprocal_ranks),
        f"NDCG@{k}": np.mean(ndcg_scores),
        f"Recall@{k}": np.mean(recall_scores),
        f"Precision@{k}": np.mean(precision_scores),
        f"MAR@{k}": np.mean(mar_scores)
    }

In [12]:
# Evaluation

final_result = []
questions = list(questionContext.keys())




In [13]:
for k in [1, 2, 3, 4, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100, 150, 200, 250, 300, 350, 400, 450, 500]:
        metrics = compute_metrics(questions, questionContext, index, k=k, max_samples=86769)
        final_result.append(metrics)
        print(f"\n🔍 Top-{k} Evaluation Metrics:")
        for metric, value in metrics.items():
            print(f"{metric}: {value:.4f}")


🔍 Top-1 Evaluation Metrics:
Accuracy@1: 25.1922
MRR@1: 0.2519
NDCG@1: 0.2519
Recall@1: 0.2519
Precision@1: 0.2519
MAR@1: 0.2519

🔍 Top-2 Evaluation Metrics:
Accuracy@2: 33.1582
MRR@2: 0.2917
NDCG@2: 0.3022
Recall@2: 0.3316
Precision@2: 0.2917
MAR@2: 0.1658

🔍 Top-3 Evaluation Metrics:
Accuracy@3: 38.2325
MRR@3: 0.3086
NDCG@3: 0.3275
Recall@3: 0.3823
Precision@3: 0.3086
MAR@3: 0.1274

🔍 Top-4 Evaluation Metrics:
Accuracy@4: 42.0196
MRR@4: 0.3181
NDCG@4: 0.3438
Recall@4: 0.4202
Precision@4: 0.3181
MAR@4: 0.1050

🔍 Top-5 Evaluation Metrics:
Accuracy@5: 45.0760
MRR@5: 0.3242
NDCG@5: 0.3557
Recall@5: 0.4508
Precision@5: 0.3242
MAR@5: 0.0902

🔍 Top-10 Evaluation Metrics:
Accuracy@10: 54.9032
MRR@10: 0.3373
NDCG@10: 0.3874
Recall@10: 0.5490
Precision@10: 0.3373
MAR@10: 0.0549

🔍 Top-15 Evaluation Metrics:
Accuracy@15: 60.7763
MRR@15: 0.3419
NDCG@15: 0.4030
Recall@15: 0.6078
Precision@15: 0.3419
MAR@15: 0.0405

🔍 Top-20 Evaluation Metrics:
Accuracy@20: 64.8722
MRR@20: 0.3443
NDCG@20: 0.4127
R

In [14]:
with open("final_result.json", "w") as f:
    json.dump(final_result, f)


🔍 Top-200 Evaluation Metrics:
Accuracy@200: 86.1506
MRR@200: 0.3491
NDCG@200: 0.4505
Recall@200: 0.8615
Precision@200: 0.3491
MAR@200: 0.0043


### 